# Kaggle Training — SE-ResNet-18 with Log-Mel Features
## ASVspoof 2019 Logical Access

**EDA-informed hyperparameters:**
- Log-Mel `[128, 251]`: n_fft=1024, hop=256, n_mels=128, fmin=20Hz, fmax=8000Hz
- SE ratio=8 → channel recalibration focuses on artifact-carrying mel bands (EDA: 6-8kHz anomaly)
- Larger batch=64 (model ~11M params, more VRAM needed)
- Mixup augmentation (α=0.2) → improves generalization to unseen attacks A07-A19


In [ ]:
import os, glob

POSSIBLE_ROOTS = [
    "/kaggle/input/asvpoof-2019-dataset",
    "/kaggle/input/asvpoof2019",
    "/kaggle/input/asvspoof-2019",
    "/kaggle/input/asvpoof-2019",
    "/kaggle/input/la-asvspoof2019",
]
DATA_ROOT = None
for p in POSSIBLE_ROOTS:
    if os.path.exists(p):
        DATA_ROOT = p
        break
if DATA_ROOT is None:
    matches = glob.glob("/kaggle/input/**/ASVspoof2019_LA_train", recursive=True)
    if matches:
        DATA_ROOT = matches[0].replace("/ASVspoof2019_LA_train", "")
if DATA_ROOT is None:
    raise RuntimeError("Dataset not found. Add ASVspoof 2019 LA dataset to this notebook.")
print(f"Dataset root: {DATA_ROOT}")


In [ ]:
import subprocess
subprocess.run(["pip", "install", "soundfile", "librosa", "-q"])


In [ ]:
import torch

CFG = {
    "sample_rate":    16000,
    "target_samples": 64000,
    "pre_emphasis":   0.97,
    "vad_top_db":     40,
    "n_fft":          1024,
    "hop_length":     256,
    "n_mels":         128,        # EDA: 128 > 80 channels for fine-grained artifact detection
    "fmin":           20,
    "fmax":           8000,
    "mel_frames":     251,

    "batch_size":     64,
    "epochs":         30,
    "lr":             5e-4,
    "lr_min":         1e-6,
    "weight_decay":   1e-4,
    "focal_alpha":    0.75,
    "focal_gamma":    2.0,
    "label_smoothing": 0.05,
    "se_reduction":   8,          # SE squeeze ratio

    "spec_t_mask":    30,
    "spec_f_mask":    15,
    "mixup_alpha":    0.2,        # Mixup for unseen attack generalization
    "dropout":        0.3,

    "seed":           42,
    "device":         "cuda" if torch.cuda.is_available() else "cpu",
    "num_workers":    2,
    "output_dir":     "/kaggle/working",
    "model_name":     "se_resnet_mel",
}

torch.manual_seed(CFG["seed"])
import numpy as np; np.random.seed(CFG["seed"])
print(f"Device: {CFG['device']}")
if CFG["device"] == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
import pandas as pd, glob

def parse_protocols(data_root):
    split_map = {"train": "ASVspoof2019_LA_train", "dev": "ASVspoof2019_LA_dev", "eval": "ASVspoof2019_LA_eval"}
    rows = []
    for split, flac_subdir in split_map.items():
        flac_dir = os.path.join(data_root, flac_subdir, "flac")
        proto_files = glob.glob(os.path.join(data_root, "**", f"*{split[:3]}*.txt"), recursive=True)
        if not proto_files:
            proto_files = glob.glob(os.path.join(data_root, "**", "*.txt"), recursive=True)
            proto_files = [f for f in proto_files if split[:3] in os.path.basename(f).lower() or split in os.path.basename(f).lower()]
        if not proto_files:
            print(f"WARNING: No protocol for {split}")
            continue
        proto_file = proto_files[0]
        with open(proto_file) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                spk, aid, _, atk, key = parts[0], parts[1], parts[2], parts[3], parts[4]
                fp = os.path.join(flac_dir, aid + ".flac")
                rows.append({"speaker_id": spk, "audio_id": aid, "attack_id": atk,
                             "key": key, "is_spoof": 1 if key == "spoof" else 0,
                             "split": split, "file_path": fp, "file_exists": os.path.exists(fp)})
    return pd.DataFrame(rows)

manifest = parse_protocols(DATA_ROOT)
for split in ["train", "dev"]:
    sub = manifest[manifest.split == split]
    bon = len(sub[sub.key == "bonafide"])
    spf = len(sub[sub.key == "spoof"])
    print(f"{split}: total={len(sub)} bon={bon} spoof={spf} ratio={spf/bon:.1f}:1 missing={len(sub[~sub.file_exists])}")


In [ ]:
import numpy as np, soundfile as sf, librosa

def load_and_process(filepath, is_training=False, cfg=CFG):
    y, sr = sf.read(filepath)
    if y.ndim > 1:
        y = y.mean(axis=1)
    if sr != cfg["sample_rate"]:
        y = librosa.resample(y.astype(np.float32), orig_sr=sr, target_sr=cfg["sample_rate"])
    y = y.astype(np.float32)
    y = np.concatenate([[y[0]], y[1:] - cfg["pre_emphasis"] * y[:-1]])
    intervals = librosa.effects.split(y=y, top_db=cfg["vad_top_db"])
    if len(intervals) > 0:
        trimmed = np.concatenate([y[s:e] for s, e in intervals])
        if len(trimmed) > 1000:
            y = trimmed
    n = len(y)
    T = cfg["target_samples"]
    if n >= T:
        start = np.random.randint(0, n - T + 1) if is_training else (n - T) // 2
        y = y[start:start + T]
    else:
        y = np.pad(y, (0, T - n), mode="wrap")
    return y / (np.max(np.abs(y)) + 1e-7)

def extract_log_mel(y, cfg=CFG):
    mel = librosa.feature.melspectrogram(
        y=y, sr=cfg["sample_rate"], n_fft=cfg["n_fft"], hop_length=cfg["hop_length"],
        n_mels=cfg["n_mels"], fmin=cfg["fmin"], fmax=cfg["fmax"]
    )
    log_mel = librosa.power_to_db(mel, ref=np.max).astype(np.float32)
    T = cfg["mel_frames"]
    if log_mel.shape[1] < T:
        log_mel = np.pad(log_mel, ((0,0),(0, T - log_mel.shape[1])), mode="edge")
    else:
        log_mel = log_mel[:, :T]
    return log_mel

# Verify
row = manifest[manifest.split == "train"].iloc[0]
y = load_and_process(row.file_path)
mel = extract_log_mel(y)
print(f"Log-Mel shape: {mel.shape}  (expected (128, 251))")
assert mel.shape == (128, 251)
print("Preprocessing verified.")


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

class MelDataset(Dataset):
    def __init__(self, df, is_training=False, cfg=CFG):
        self.df = df.reset_index(drop=True)
        self.is_training = is_training
        self.cfg = cfg

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            y = load_and_process(row.file_path, self.is_training, self.cfg)
            x = extract_log_mel(y, self.cfg)
        except Exception:
            x = np.zeros((self.cfg["n_mels"], self.cfg["mel_frames"]), dtype=np.float32)
        if self.is_training:
            if np.random.rand() < 0.5:
                t = np.random.randint(1, self.cfg["spec_t_mask"])
                t0 = np.random.randint(0, max(1, self.cfg["mel_frames"] - t))
                x[:, t0:t0+t] = x.mean()
            if np.random.rand() < 0.5:
                f = np.random.randint(1, self.cfg["spec_f_mask"])
                f0 = np.random.randint(0, max(1, self.cfg["n_mels"] - f))
                x[f0:f0+f, :] = x.mean()
        return torch.from_numpy(x).unsqueeze(0), torch.tensor(int(row.is_spoof), dtype=torch.long)

train_df = manifest[manifest.split == "train"].reset_index(drop=True)
dev_df   = manifest[manifest.split == "dev"].reset_index(drop=True)

labels = train_df.is_spoof.values
class_counts = np.bincount(labels)
sample_weights = torch.FloatTensor((1.0 / class_counts)[labels])
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_ds = MelDataset(train_df, is_training=True)
dev_ds   = MelDataset(dev_df, is_training=False)
train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], sampler=sampler,
                          num_workers=CFG["num_workers"], pin_memory=True)
dev_loader   = DataLoader(dev_ds, batch_size=CFG["batch_size"], shuffle=False,
                          num_workers=CFG["num_workers"], pin_memory=True)
print(f"Train: {len(train_ds)} | Dev: {len(dev_ds)} | Batches/epoch: {len(train_loader)}")


In [ ]:
import torch.nn as nn

class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(channels, channels // reduction), nn.ReLU(),
            nn.Linear(channels // reduction, channels), nn.Sigmoid()
        )
    def forward(self, x):
        s = self.se(x).view(x.size(0), x.size(1), 1, 1)
        return x * s

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, reduction=8):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False), nn.BatchNorm2d(out_ch),
        )
        self.se = SEBlock(out_ch, reduction)
        self.downsample = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, stride, bias=False), nn.BatchNorm2d(out_ch)
        ) if stride != 1 or in_ch != out_ch else nn.Identity()
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(self.se(self.conv(x)) + self.downsample(x))

class SEResNet18(nn.Module):
    def __init__(self, reduction=8):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(1, 32, 7, 2, 3, bias=False), nn.BatchNorm2d(32),
            nn.ReLU(), nn.MaxPool2d(3, 2, 1)
        )
        self.layer1 = nn.Sequential(ResBlock(32, 32, reduction=reduction), ResBlock(32, 32, reduction=reduction))
        self.layer2 = nn.Sequential(ResBlock(32, 64, stride=2, reduction=reduction), ResBlock(64, 64, reduction=reduction))
        self.layer3 = nn.Sequential(ResBlock(64, 128, stride=2, reduction=reduction), ResBlock(128, 128, reduction=reduction))
        self.layer4 = nn.Sequential(ResBlock(128, 256, stride=2, reduction=reduction), ResBlock(256, 256, reduction=reduction))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Flatten(), nn.Dropout(CFG["dropout"]),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return self.head(self.pool(x))

device = torch.device(CFG["device"])
model = SEResNet18(reduction=CFG["se_reduction"]).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"SE-ResNet-18 parameters: {n_params:,}")
with torch.no_grad():
    dummy = torch.zeros(2, 1, 128, 251).to(device)
    out = model(dummy)
    print(f"Output shape: {out.shape}  (expected [2, 2])")


In [ ]:
import torch.nn.functional as F, time, json
from sklearn.metrics import roc_auc_score

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, label_smoothing=0.05):
        super().__init__()
        self.alpha, self.gamma, self.label_smoothing = alpha, gamma, label_smoothing
    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, reduction="none", label_smoothing=self.label_smoothing)
        pt = torch.exp(-ce)
        alpha_t = torch.where(targets == 1, self.alpha, 1.0 - self.alpha)
        return (alpha_t * (1 - pt) ** self.gamma * ce).mean()

def compute_eer(y_true, y_score):
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(y_true, y_score, pos_label=1)
    fnr = 1 - tpr
    idx = np.nanargmin(np.abs(fpr - fnr))
    return float((fpr[idx] + fnr[idx]) / 2)

def mixup_batch(xb, yb, alpha=0.2):
    if alpha <= 0 or np.random.rand() > 0.5:
        return xb, yb, yb, 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(xb.size(0))
    return lam * xb + (1 - lam) * xb[idx], yb, yb[idx], lam

criterion = FocalLoss(CFG["focal_alpha"], CFG["focal_gamma"], CFG["label_smoothing"])
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=CFG["lr_min"])
scaler = torch.cuda.amp.GradScaler(enabled=(CFG["device"] == "cuda"))

best_eer, best_auc = float("inf"), 0.0
history = []
save_path = os.path.join(CFG["output_dir"], f"{CFG['model_name']}_best.pth")

for epoch in range(1, CFG["epochs"] + 1):
    model.train()
    total_loss, t0 = 0.0, time.time()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        xb_m, ya, yb_m, lam = mixup_batch(xb, yb, CFG["mixup_alpha"])
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(CFG["device"] == "cuda")):
            logits = model(xb_m)
            loss = lam * criterion(logits, ya) + (1 - lam) * criterion(logits, yb_m)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * xb.size(0)
    train_loss = total_loss / len(train_ds)
    scheduler.step(epoch)

    model.eval()
    all_probs, all_targets = [], []
    with torch.no_grad():
        for xb, yb in dev_loader:
            with torch.cuda.amp.autocast(enabled=(CFG["device"] == "cuda")):
                logits = model(xb.to(device))
            all_probs.append(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
            all_targets.append(yb.numpy())
    y_prob = np.concatenate(all_probs)
    y_true = np.concatenate(all_targets)
    eer = compute_eer(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    acc = ((y_prob >= 0.5) == y_true).mean()

    print(f"Ep {epoch:02d}/{CFG['epochs']} | loss={train_loss:.4f} | EER={eer*100:.2f}% | AUC={auc:.4f} | acc={acc*100:.1f}% | {time.time()-t0:.0f}s")
    history.append({"epoch": epoch, "train_loss": round(train_loss,6), "val_eer": round(eer,6), "val_auc": round(auc,6)})

    if eer < best_eer:
        best_eer, best_auc = eer, auc
        torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                    "eer": best_eer, "auc": best_auc, "cfg": CFG}, save_path)
        print(f"  >>> Best: EER={best_eer*100:.2f}% AUC={best_auc:.4f}")

json.dump(history, open(os.path.join(CFG["output_dir"], f"{CFG['model_name']}_history.json"), "w"), indent=2)
print(f"\nBest EER: {best_eer*100:.2f}% | Best AUC: {best_auc:.4f}")


In [ ]:
import matplotlib.pyplot as plt

epochs_list = [h["epoch"] for h in history]
eers = [h["val_eer"]*100 for h in history]
aucs = [h["val_auc"] for h in history]
losses = [h["train_loss"] for h in history]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(epochs_list, losses, color="#3498db", lw=2)
axes[0].set_title("Training Loss"); axes[0].set_xlabel("Epoch")

axes[1].plot(epochs_list, eers, color="#e74c3c", lw=2, marker="o", ms=4)
axes[1].axhline(min(eers), color="gray", ls="--", label=f"Best: {min(eers):.2f}%")
axes[1].set_title("Dev EER (%)"); axes[1].legend()

axes[2].plot(epochs_list, aucs, color="#2ecc71", lw=2, marker="o", ms=4)
axes[2].axhline(max(aucs), color="gray", ls="--", label=f"Best: {max(aucs):.4f}")
axes[2].set_title("Dev AUC"); axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(CFG["output_dir"], f"{CFG['model_name']}_curve.png"), dpi=150)
plt.show()
print(f"Best EER: {best_eer*100:.2f}% | AUC: {best_auc:.4f} | Saved: {save_path}")
